In [34]:
import pandas as pd
import numpy as np

def compute_ess(df, target, group):
    """
    Compute ESS for a clustered dataset.
    
    Parameters
    ----------
    df     : DataFrame
    target : column name of the continuous target (e.g. predicted PD)
    group  : column name of the group identifier (e.g. firm, industry)

    Calculated
    __________
    N      : Number of groups
    M      : Total observations
    M_j    : Observations in group j
    
    Returns
    -------
    dict with rho, d_eff, DEFF, ESS, coefficient of variation, and basic counts
    """
    x = df[target].values
    groups = df[group].values
    
    M = len(x)
    grand_mean = x.mean()
    
    # Group sizes and means
    grp = df.groupby(group)[target]
    M_j = grp.count()
    x_j = grp.mean()
    cv_N = df.groupby(group)[target].mean().std() / df.groupby(group)[target].mean().mean()
    
    N = len(M_j)
    w_j = M_j / M                                        # size weights
    
    # Variance decomposition — unequal group sizes
    between = (w_j * (x_j - grand_mean)**2).sum()
    within  = (w_j * grp.var(ddof=0)).sum()
    total   = between + within
    
    # ICC
    rho = between / total
    
    # Kish effective cluster size
    d_eff = (M_j**2).sum() / M
    
    # Design effect and ESS
    DEFF = 1 + (d_eff - 1) * rho
    ESS  = M / DEFF
    
    return {
        "Num Groups (N)"   : N,
        "Total Obs (M)"    : M,
        "Obs per Group"      : M / N,
        "d_eff"      : d_eff,
        "rho"        : rho,
        "DEFF"       : DEFF,
        "ESS"        : ESS,
        "ESS/M"      : ESS / M,
        "ESS/N"      : ESS / N,
        "Coef Var"   : cv_N,
    }


